In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vasukipatel/face-recognition-dataset")

print("Path to dataset files:", path)

100%|██████████| 726M/726M [00:08<00:00, 88.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/vasukipatel/face-recognition-dataset/versions/1


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
path = "/root/.cache/kagglehub/datasets/vasukipatel/face-recognition-dataset/versions/1"
# 1. Model Definition
class FaceCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)
# 2. Custom Binary Dataset Loader
class BinaryFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_person_name='Akshay Kumar'):
        self.root_dir = root_dir
        self.transform = transform
        self.target_person_name = target_person_name
        self.samples = []

        if not os.path.isdir(root_dir):
            raise FileNotFoundError(f"Directory not found: {root_dir}")

        first_level_items = os.listdir(root_dir)
        is_nested_structure = False
        if first_level_items:
            first_item_path = os.path.join(root_dir, first_level_items[0])
            if os.path.isdir(first_item_path):
                is_nested_structure = True

        if is_nested_structure:
            for class_folder in first_level_items:
                class_path = os.path.join(root_dir, class_folder)
                if os.path.isdir(class_path):
                    for filename in os.listdir(class_path):
                        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                            label = 0 if class_folder == self.target_person_name else 1
                            self.samples.append((os.path.join(class_path, filename), label))
        else:
            for filename in first_level_items:
                if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                    person_name = filename.split('_')[0]
                    label = 0 if person_name == self.target_person_name else 1
                    self.samples.append((os.path.join(root_dir, filename), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label
# 3. Image Preprocessing Transformation
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])
# 4. Dataset & DataLoader Setup
num_classes = 2

train_dataset = BinaryFaceDataset(
    root_dir=os.path.join(path, "Original Images", "Original Images"),
    transform=transform,
    target_person_name='Akshay Kumar'
)

test_dataset = BinaryFaceDataset(
    root_dir=os.path.join(path, "Faces", "Faces"),
    transform=transform,
    target_person_name='Akshay Kumar'
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)
# 5. Device Configuration & Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FaceCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# 6. Model Training Loop
epochs = 10
print("--- Starting Training ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {running_loss/len(train_loader):.4f}")
# Save Model Weights
torch.save(model.state_dict(), "face_model.pth")
print("Model Saved as 'face_model.pth'!")
# 7. Model Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
# 8. Interactive Single Image Prediction (Added at the end)
print("\n--- Single Image Prediction ---")
classes = ["Akshay Kumar", "Unknown / Not Akshay"]

image_path = input("Enter image path for prediction: ").strip()

if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image_tensor)
        _, prediction = torch.max(output, 1)

    print(f"\nPrediction Result: {classes[prediction.item()]}")
else:
    print("Error: Invalid image path!")

--- Starting Training ---
Epoch 1/10 Loss: 0.1348
Epoch 2/10 Loss: 0.1150
Epoch 3/10 Loss: 0.1043
Epoch 4/10 Loss: 0.1040
Epoch 5/10 Loss: 0.0903
Epoch 6/10 Loss: 0.0956
Epoch 7/10 Loss: 0.0908
Epoch 8/10 Loss: 0.0805
Epoch 9/10 Loss: 0.0778
Epoch 10/10 Loss: 0.0764
Model Saved as 'face_model.pth'!
Test Accuracy: 98.05%

--- Single Image Prediction ---
Enter image path for prediction: /cute-alien-wearing-hoodie-jacket-cartoon-vector-icon-illustration-science-technology-isolated-flat_138676-13026.avif

Prediction Result: Unknown / Not Akshay


In [ ]:
print("\n--- Single Image Prediction ---")
classes = ["Akshay Kumar", "Unknown / Not Akshay"]

image_path = input("Enter image path for prediction: ").strip()

if os.path.exists(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image_tensor)
        _, prediction = torch.max(output, 1)

    print(f"\nPrediction Result: {classes[prediction.item()]}")
else:
    print("Error: Invalid image path!")


--- Single Image Prediction ---
Enter image path for prediction: print("\n--- Single Image Prediction ---") classes = ["Akshay Kumar", "Unknown / Not Akshay"]  image_path = input("Enter image path for prediction: ").strip()  if os.path.exists(image_path):     image = Image.open(image_path).convert("RGB")     image_tensor = transform(image).unsqueeze(0).to(device)      with torch.no_grad():         output = model(
Error: Invalid image path!
